<i>

## Exercise Project 2 -- Loan Data

<h2><strong style='color:orange ;'> Gradio Graphical User Interface</strong></h2>


In this model application, we'll create a  GUI that allows the user give their personal information and loan proposition details to a web-page (or cell!)<strong> to see if they are approved or denied for a loan.</strong>
We'll be using `gradio` for this application. 

---

### Code:

In [1]:
# Imports
import pandas as pd
import gradio as gr
import joblib

# Load the Linear Regression model.
lm = joblib.load("models/ex2_main.pkl")

# This FUNCTION runs the model trained in ex2_01_main.ipynb for the Gradio GUI
def predict(
        select_person_age, select_person_income, select_person_emp_exp,
        select_loan_amnt, select_loan_int_rate,
        select_cb_person_cred_hist_length, previous_loan_defaults_on_file,
        select_person_home_ownership, select_loan_intent
    ):

    # First, we convert the user's function parameters into a list.
    # Notice some are missing. We're going to infer the rest.
    tester_row = {
    'person_age': select_person_age,
    'person_income': select_person_income,
    'person_emp_exp': select_person_emp_exp,
    'loan_amnt': select_loan_amnt,
    'loan_int_rate': select_loan_int_rate,
    'cb_person_cred_hist_length': select_cb_person_cred_hist_length,
    'previous_loan_defaults_on_file': previous_loan_defaults_on_file,
    }

    # Infer the loan percentage of income without requiring the user to input anything
    try:
        loan_percent_income =  select_loan_amnt / select_person_income
        tester_row["loan_percent_income"] = loan_percent_income
    except ZeroDivisionError:
        tester_row["loan_percent_income"] = 0.00
    

    # Lists of features to iterate over
    home_options_list=["Own", "Mortgage"]
    loan_options_list = [
        "DEBTCONSOLIDATION", "EDUCATION",
        "HOMEIMPROVEMENT", "MEDICAL", "PERSONAL"
    ]   

    # --INFER THE CATEGORICAL OPTIONS -- 
    # If the user selected this option...
    # The value is 1, else 0
    for option in home_options_list:
        if select_person_home_ownership == option:
            tester_row[f"person_home_ownership_{option}"] = 1
        else:
            tester_row[f"person_home_ownership_{option}"] = 0

    # If the user selected this option...
    # The value is 1, else 0
    for option in loan_options_list:
        if select_loan_intent == option:
            tester_row[f"loan_intent_{option}"] = 1
        else:
            tester_row[f"loan_intent_{option}"] = 0
                        
    # Then, we transform it into a dataframe.
    tester_row = pd.DataFrame([tester_row])

    # Predict and round the result
    result = lm.predict(tester_row)[0]
    
    if result == 1:
        result = "Loan Approved"
    else:
        result = "Loan Rejected"
    
    return result

#  -- FUNCTION PARAMETER GUIDE --
#  select_person_age,
#  select_person_income,
#  select_person_emp_exp,
#  select_loan_amnt,
#  select_loan_int_rate,
#  select_cb_person_cred_hist_length,
#  previous_loan_defaults_on_file,
#  select_person_home_ownership,
#  select_loan_intent

# Some quick tests to make sure it's working correctly.
assert predict(40,200000,10,15000,11,5,0, "Own", "DEBTCONSOLIDATION", ) == "Loan Approved"
assert predict(40,10000,10,15000,11,5,0, "Own", "DEBTCONSOLIDATION", ) == "Loan Rejected"
assert predict(40,200000,10, 100000,11,5,0, "Own", "DEBTCONSOLIDATION", ) == "Loan Rejected"
assert predict(40,200000,10, 100000,0,5,4, "Own", "DEBTCONSOLIDATION", ) == "Loan Rejected"

c:\Users\kykis\Intro_ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\kykis\Intro_ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\kykis\Intro_ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\kykis\Intro_ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [2]:
# Use Gradio Blocks to create a GUI! This should show up in the output cell.
with gr.Blocks() as house_lot_prediction:

    gr.Markdown("## 💰💰 Bank Loan Assessor 💰💰")
    gr.Markdown("### Enter Your Personal Details Below:")

    # These selects give a number of options and questions for the user.
    # (Similar to javascript!)
    select_person_age = gr.Number(label="Age")
    select_person_income = gr.Number(label="Personal Income")
    select_person_emp_exp = gr.Number(label="How much job experience in your field")
    select_loan_amnt = gr.Number(label="Loan Amount")
    select_loan_int_rate = gr.Number(label="Interest rate percentage")
    select_cb_person_cred_hist_length = gr.Number(label="Amount of credit history in years")
    previous_loan_defaults_on_file = gr.Number(label="Number of loan defaults")
    select_person_home_ownership = gr.Radio(
        choices=["Own", "Rent", "Mortgage", "Other"],
        label="Your living situation"
        )
    select_loan_intent = gr.Radio(
        choices = ["DEBTCONSOLIDATION", "EDUCATION",
                   "HOMEIMPROVEMENT", "MEDICAL", "PERSONAL"
        ],
        label="Loan Purpose"
    )
    
    # -- some notes (if you're noticing some selects may be missing) -- 
    # select_loan_percent_income     -- Calculated in the function
    # select_person_home_ownership   -- Gets interpreted by seeing if the user selected that option
    # select_loan_intent             -- Gets interpreted by seeing if the user selected that option
    
    # The radios get unpacked and encoded in the function itself.

    gr.Markdown("### Prediction")

    # Prediction output and submit button
    outputs = gr.Text(label="Home Price")
    submit = gr.Button("Predict Price")

    # Click action functionality
    submit.click(
        fn=predict,
        inputs=[
        select_person_age, select_person_income, select_person_emp_exp,
        select_loan_amnt, select_loan_int_rate,
        select_cb_person_cred_hist_length, previous_loan_defaults_on_file,
        select_person_home_ownership, select_loan_intent
            ],
        outputs=outputs
    )

house_lot_prediction.launch()
# Click the URL link to have it show up in your browser. (looks a little nicer)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\kykis\Intro_ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
